In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
data = pd.read_csv("dataset (1).csv")
data.head()

In [ ]:
features = ["danceability", "energy", "tempo", "loudness", "valence"]

data_clean = data[features + ["popularity"]].dropna()
X = data_clean[features].values          # numpy array
y = data_clean["popularity"].values       # numpy array

In [ ]:
# Manual train/test split (80/20)
np.random.seed(42)
idx = np.random.permutation(len(X))
split = int(len(X) * 0.8)
train_idx, test_idx = idx[:split], idx[split:]

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

In [ ]:
# Manual standardization: fit on train, apply to test
X_mean = X_train.mean(axis=0)
X_std  = X_train.std(axis=0)

X_train = (X_train - X_mean) / X_std
X_test  = (X_test  - X_mean) / X_std

In [ ]:
class ManualMLP:
    """
    Multi-Layer Perceptron for regression, implemented from scratch using NumPy.
    Architecture : input -> hidden layers (ReLU) -> output (linear)
    Optimizer     : mini-batch stochastic gradient descent
    """

    def __init__(self, layer_sizes, learning_rate=0.001,
                 max_iter=500, batch_size=256, random_state=42):
        np.random.seed(random_state)
        self.layer_sizes  = layer_sizes
        self.lr           = learning_rate
        self.max_iter     = max_iter
        self.batch_size   = batch_size
        self.loss_history = []

        # He initialization (suited for ReLU activations)
        self.weights = []
        self.biases  = []
        for i in range(len(layer_sizes) - 1):
            w = np.random.randn(layer_sizes[i], layer_sizes[i+1])                 * np.sqrt(2.0 / layer_sizes[i])
            b = np.zeros((1, layer_sizes[i+1]))
            self.weights.append(w)
            self.biases.append(b)

    # Activation functions
    def _relu(self, z):
        return np.maximum(0, z)

    def _relu_grad(self, z):
        return (z > 0).astype(float)

    # Forward pass
    def _forward(self, X):
        self._activations = [X]
        self._z_vals = []
        for i, (w, b) in enumerate(zip(self.weights, self.biases)):
            z = self._activations[-1] @ w + b
            self._z_vals.append(z)
            # ReLU on hidden layers; linear activation on output layer
            a = self._relu(z) if i < len(self.weights) - 1 else z
            self._activations.append(a)
        return self._activations[-1]

    # Backward pass (backpropagation)
    def _backward(self, y_true):
        m = y_true.shape[0]
        y_true = y_true.reshape(-1, 1)

        # Gradient of MSE loss w.r.t. output layer
        delta = 2 * (self._activations[-1] - y_true) / m

        grad_w = [None] * len(self.weights)
        grad_b = [None] * len(self.weights)

        for i in reversed(range(len(self.weights))):
            grad_w[i] = self._activations[i].T @ delta
            grad_b[i] = delta.sum(axis=0, keepdims=True)
            if i > 0:
                delta = (delta @ self.weights[i].T) * self._relu_grad(self._z_vals[i-1])

        # Gradient descent weight update
        for i in range(len(self.weights)):
            self.weights[i] -= self.lr * grad_w[i]
            self.biases[i]  -= self.lr * grad_b[i]

    # Training loop
    def fit(self, X, y):
        for epoch in range(self.max_iter):
            perm = np.random.permutation(X.shape[0])
            X_s, y_s = X[perm], y[perm]

            for start in range(0, X.shape[0], self.batch_size):
                Xb = X_s[start : start + self.batch_size]
                yb = y_s[start : start + self.batch_size]
                self._forward(Xb)
                self._backward(yb)

            # Record full-set MSE loss every epoch
            loss = np.mean((self._forward(X).flatten() - y) ** 2)
            self.loss_history.append(loss)

            if (epoch + 1) % 100 == 0:
                print(f"Epoch {epoch+1:4d}/{self.max_iter}  MSE Loss: {loss:.4f}")

    # Prediction
    def predict(self, X):
        return self._forward(X).flatten()


In [ ]:
model = ManualMLP(
    layer_sizes   = [5, 64, 32, 1],
    learning_rate = 0.001,
    max_iter      = 500,
    batch_size    = 256,
    random_state  = 42
)

model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
# Metrics computed manually with NumPy
rmse   = np.sqrt(np.mean((y_test - y_pred) ** 2))
mae    = np.mean(np.abs(y_test - y_pred))
ss_res = np.sum((y_test - y_pred) ** 2)
ss_tot = np.sum((y_test - y_test.mean()) ** 2)
r2     = 1 - ss_res / ss_tot

print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"R2:   {r2:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Popularity Distribution
axes[0].hist(y, bins=40, color="steelblue", edgecolor="white")
axes[0].set_xlabel("Popularity")
axes[0].set_ylabel("Count")
axes[0].set_title("Popularity Distribution")

# Plot 2: Prediction vs Actual
axes[1].scatter(y_test, y_pred, alpha=0.3, s=10, color="steelblue")
axes[1].plot([0, 100], [0, 100], "r--", linewidth=1.5, label="Ideal")
axes[1].set_xlabel("Actual Popularity")
axes[1].set_ylabel("Predicted Popularity")
axes[1].set_title("Prediction vs Actual")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Plot 3: Training Loss Curve
plt.figure(figsize=(8, 4))
plt.plot(model.loss_history, color="steelblue", linewidth=1.5)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Training Loss Curve")
plt.tight_layout()
plt.show()